# Part 2 Test-Time Scaling

This notebook loads an AIME 2024 dataset, runs a model on each problem, extracts an AIME-style final answer, and grades the outputs.

In [1]:
import re

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

!pip install vllm==0.20.0

from vllm import LLM, SamplingParams


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

ImportError: cannot import name 'AutoModelForCausalLM' from 'transformers' (unknown location)

In [2]:
MODEL_NAME = "Qwen/Qwen3-4B"
# or allenai/Olmo-3-7B-Thinking
DATASET_NAME = "OpenRLHF/aime-2024"
MAX_NEW_TOKENS = 4096 #512

## Loading the model and the data

In [ ]:
# Using vLLM instead for faster inference
dtype = "float16" if torch.cuda.is_available() else "float32"

llm       = LLM(model=MODEL_NAME, dtype=dtype, max_model_len=32768)
tokenizer = llm.get_tokenizer()  # no need for separate AutoTokenizer

dataset = load_dataset(DATASET_NAME, split="train")

In [3]:
# device = "cuda" if torch.cuda.is_available() else "cpu"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16 if device == "cuda" else torch.float32,
# )

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# dataset = load_dataset(DATASET_NAME, split="train")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/371 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

## Evaluation helpers

In [4]:
def extract_thinking_trace(text):
    match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    return match.group(1).strip() if match else ""


def strip_thinking_trace(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|begin_of_thought\|>.*?<\|end_of_thought\|>", "", text, flags=re.DOTALL)
    return text.strip()


def extract_answer(text: str, mode="exact_match") -> int | None:
    """Extract an AIME-style integer answer from a model completion."""
    answer_text = strip_thinking_trace(text)
    if not answer_text:
        if mode == "exact_match":
            return None
        else:
            answer_text = text  # fall back to full text


    # 1. Boxed LaTeX answer: \boxed{123}
    if mode == "exact_match":
        boxed = re.findall(r"\\boxed\{(\d+)\}", answer_text)
        if boxed:
            val = int(boxed[-1])
            return val
        else:
            return None

    elif mode == "flexible_extract":
        # 2. "The answer is N" or "answer: N" patterns
        patterns = [
            r"(?:the\s+)?answer\s+is\s+[:\s]*(\d+)",
            r"answer[:\s]+(\d+)",
            r"=\s*(\d+)\s*$",
            r"(?:therefore|thus|so),?\s+(\d+)\s*(?:\.|$)",
        ]
        for pattern in patterns:
            matches = re.findall(pattern, answer_text, re.IGNORECASE)
            if matches:
                val = int(matches[-1])
                return val

        # 3. Last integer in [0, 999] in the answer portion
        integers = re.findall(r"\b(\d{1,3})\b", answer_text)
        for candidate in reversed(integers):
            val = int(candidate)
            return val
        return None


## 2.1 Warm-Up

You can also explore using vLLM to speed up inference!

In [6]:
## DEPRECIATED: too slow!
ENABLE_THINKING = True  # Ensure thinking is enabled for this analysis
ANSWER_MODE = "flexible_extract"

model.to(device)
model.eval()

records = []

for i, example in enumerate(dataset):
    problem = example["prompt"][0]["content"]
    gold_answer = int(example["label"])

    messages = [{"role": "system", "content": "You are a careful competition math assistant.  Always output your final answer in \\boxed{}."},
                {"role": "user", "content": problem}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    # Decode only the newly generated tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    model_output = tokenizer.decode(generated_ids, skip_special_tokens=False)

    # Debug: print raw tags on first example to confirm tag format
    if i == 0:
        print("RAW OUTPUT SNIPPET:", repr(model_output[:300]))

    extracted = extract_answer(model_output, mode=ANSWER_MODE)
    if extracted is not None:
        correct = extracted == gold_answer
    else:
        correct = False

    # Extract thinking trace and calculate its token length
    extracted_thinking = extract_thinking_trace(model_output)
    thinking_tokens = tokenizer.encode(extracted_thinking, add_special_tokens=False)
    thinking_length = len(thinking_tokens)

    records.append({
        "problem": problem,
        "gold_answer": gold_answer,
        "model_output": model_output,
        "extracted_answer": extracted,
        "correct": correct,
        "thinking_length": thinking_length, # Add thinking length to records
    })

    print(f"[{i+1}/{len(dataset)}] gold={gold_answer} pred={extracted} correct={correct} thinking_len={thinking_length}")

results_df = pd.DataFrame(records)

print("\nDistribution of Thinking Lengths (in tokens):")
print(results_df["thinking_length"].describe())

results_df

RAW OUTPUT SNIPPET: '<think>\nOkay, so I need to find the largest possible real part of the expression $(75 + 117i)z + \\frac{96 + 144i}{z}$ where $z$ is a complex number with $|z| = 4$. Hmm, let me think about how to approach this.\n\nFirst, since $z$ is a complex number with modulus 4, I can represent $z$ in polar form. T'
[1/30] gold=540 pred=4 correct=False thinking_len=3949


KeyboardInterrupt: 

In [ ]:
prompts, gold_answers, problems = [], [], []
for example in dataset:
    problem = example["prompt"][0]["content"]
    problems.append(problem)
    gold_answers.append(int(example["label"]))
    prompts.append(tokenizer.apply_chat_template(
        [{"role": "system", "content": "You are a careful competition math assistant. Always output your final answer in \\boxed{}."},
         {"role": "user",   "content": problem}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True,
    ))

sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, detokenize=True)
outputs  = llm.generate(prompts, sampling_params=sampling)

print("RAW OUTPUT SNIPPET:", repr(outputs[0].outputs[0].text[:300]))

records = []
for i, (out, gold, problem) in enumerate(zip(outputs, gold_answers, problems)):
    model_output = out.outputs[0].text
    extracted    = extract_answer(model_output, mode=ANSWER_MODE)
    correct      = extracted == gold if extracted is not None else False
    thinking     = extract_thinking_trace(model_output)
    think_len    = len(tokenizer.encode(thinking, add_special_tokens=False))

    records.append(dict(problem=problem, gold_answer=gold, model_output=model_output,
                        extracted_answer=extracted, correct=correct, thinking_length=think_len))
    print(f"[{i+1}/{len(dataset)}] gold={gold} pred={extracted} correct={correct} thinking_len={think_len}")

results_df = pd.DataFrame(records)
print("\nDistribution of Thinking Lengths (in tokens):")
print(results_df["thinking_length"].describe())

prompts_no_thinking = []
for i, example in enumerate(dataset):
    prompts_no_thinking.append(tokenizer.apply_chat_template(
        [{"role": "system", "content": "You are a careful competition math assistant. Always output your final answer in \\boxed{}."},
         {"role": "user",   "content": problems[i]}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    ))

sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, detokenize=True)
outputs_no_thinking  = llm.generate(prompts, sampling_params=sampling)

print("RAW OUTPUT SNIPPET:", repr(outputs_no_thinking[0].outputs_no_thinking[0].text[:300]))

records_no_thinking = []
for i, (out, gold, problem) in enumerate(zip(outputs_no_thinking, gold_answers, problems)):
    model_output = out.outputs[0].text
    extracted    = extract_answer(model_output, mode=ANSWER_MODE)
    correct      = extracted == gold if extracted is not None else False
    thinking     = extract_thinking_trace(model_output)
    think_len    = len(tokenizer.encode(thinking, add_special_tokens=False))

    records_no_thinking.append(dict(problem=problem, gold_answer=gold, model_output=model_output,
                        extracted_answer=extracted, correct=correct, thinking_length=think_len))
    print(f"[{i+1}/{len(dataset)}] gold={gold} pred={extracted} correct={correct} thinking_len={think_len}")

results_df_no_thinking = pd.DataFrame(records_no_thinking)
print("\nDistribution of Thinking Lengths (in tokens):")
print(results_df_no_thinking["thinking_length"].describe())

In [ ]:
print(f'Thinking-enabled accuracy: {results_df["correct"].mean()}')
print(f'No-thinking accuracy: {results_df_no_thinking["correct"].mean()}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

# Histogram for Thinking-enabled mode
sns.histplot(results_df['thinking_length'], bins=50, color='skyblue', label='Thinking Enabled', kde=True)

# Histogram for No-thinking mode (will likely be all 0s)
sns.histplot(results_df_no_thinking['thinking_length'], bins=50, color='salmon', label='No Thinking', kde=True)

plt.title('Distribution of Thinking Lengths (in Tokens) for Both Inference Modes')
plt.xlabel('Thinking Length (Tokens)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True)
plt.show()

## 2.2 Scaling Experiments

In [ ]:
def run_one(llm, tokenizer, problem, budget, strategy="wait",
            do_sample=False, temperature=0.6, top_k=50, top_p=0.95):
    end_ids, end_tag = think_end_ids(tokenizer)
    prompt   = make_prompt(problem, tokenizer)
    sampling = SamplingParams(
        temperature      = temperature if do_sample else 0.0,
        top_k            = top_k  if do_sample else -1,
        top_p            = top_p  if do_sample else 1.0,
        max_tokens       = 1,
        detokenize       = False,
    )

    token_ids = tokenizer.encode(prompt)

    # ── thinking loop ─────────────────────────────────────────────────────────
    while True:
        think_len = len(token_ids) - len(tokenizer.encode(prompt))
        out       = llm.generate(prompt_token_ids=[token_ids], sampling_params=sampling)
        next_tok  = out[0].outputs[0].token_ids[0]

        if think_len >= budget:
            token_ids += end_ids
            break
        elif next_tok in end_ids and strategy == "wait":
            wait_id = tokenizer.encode("Wait", add_special_tokens=False)[0]
            token_ids.append(wait_id)
        else:
            token_ids.append(next_tok)
            if next_tok in end_ids:
                break

    # ── generate final answer ─────────────────────────────────────────────────
    ans_sampling = SamplingParams(
        temperature = temperature if do_sample else 0.0,
        top_k       = top_k  if do_sample else -1,
        top_p       = top_p  if do_sample else 1.0,
        max_tokens  = MAX_ANS_TOKENS,
    )
    out      = llm.generate(prompt_token_ids=[token_ids], sampling_params=ans_sampling)
    decoded  = out[0].outputs[0].text
    full_out = tokenizer.decode(token_ids) + decoded

    thinking  = extract_thinking_trace(full_out)
    think_tok = len(tokenizer.encode(thinking, add_special_tokens=False))
    total_tok = len(token_ids) - len(tokenizer.encode(prompt)) + len(out[0].outputs[0].token_ids)
    return full_out, think_tok, total_tok
